#Initializing silver layer

In [0]:
from pyspark.sql import functions as fs


In [0]:
df = spark.table("`cafe-sales`.bronze.caffee_sales")
df.display()

In [0]:
# Normalize error values handling ERROR/Unknown/null
df = df.withColumn("Item", 
                    fs.when(fs.col("Item").isNull() |
                    fs.col("Item").isin("ERROR", "UNKNOWN"),
                    "Unknown").otherwise(fs.col("Item")))
df.select("Item").distinct().display()

In [0]:
# Normalize error values handling ERROR/Unknown/null
df = df.withColumn("Payment_Method", 
                    fs.when(fs.col("Payment_Method").isNull() |
                    fs.col("Payment_Method").isin("ERROR", "UNKNOWN"),
                    "Unknown").otherwise(fs.col("Payment_Method")))
df.select("Payment_Method").distinct().display()

In [0]:
#Normalizing Quantity, Total_Spent, Price_Per_Unit errors and casting column types to double
df = df.withColumn("Quantity", 
                    fs.when(fs.col("Quantity").isNull() |
                    fs.col("Quantity").isin("ERROR", "UNKNOWN"),
                    None).otherwise(fs.col("Quantity")))
df = df.withColumn("Quantity", fs.col("Quantity").cast("double"))

df = df.withColumn("Total_Spent", 
                    fs.when(fs.col("Total_Spent").isNull() |
                    fs.col("Total_Spent").isin("ERROR", "UNKNOWN"),
                    None).otherwise(fs.col("Total_Spent")))
df = df.withColumn("Total_Spent", fs.col("Total_Spent").cast("double"))

df = df.withColumn("Price_Per_Unit", 
                    fs.when(fs.col("Price_Per_Unit").isNull() |
                    fs.col("Price_Per_Unit").isin("ERROR", "UNKNOWN"),
                    None).otherwise(fs.col("Price_Per_Unit")))
df = df.withColumn("Price_Per_Unit", fs.col("Price_Per_Unit").cast("double"))

In [0]:
# Defining product prices
df = df.withColumn("New_Price_Per_Unit", 
              fs.when((fs.col("Quantity") != 0) & 
                      (fs.col("Total_Spent") != 0), 
                      fs.col("Total_Spent") / fs.col("Quantity"))
              .otherwise(fs.col("Price_Per_Unit")))

df = df.withColumn("Price_Per_Unit", fs.col("New_Price_Per_Unit"))
df = df.drop("New_Price_Per_Unit")
df.where(fs.col("Price_Per_Unit").isNull()).display()


In [0]:
# Normalizing Quantity, Total_Spent errors
df = df.withColumn(
    "Quantity",
    fs.when(
        fs.col("Quantity").isNull(),
        fs.col("Total_Spent") / fs.col("Price_Per_Unit")
    ).otherwise(fs.col("Quantity"))
)

df = df.withColumn(
    "Total_Spent",
    fs.when(
        fs.col("Total_Spent").isNull(),
        fs.col("Quantity") * fs.col("Price_Per_Unit")
    ).otherwise(fs.col("Total_Spent"))
)

In [0]:
# Normalize error values handling ERROR/Unknown/null
df = df.withColumn("Location", 
                    fs.when(fs.col("Location").isNull() |
                    fs.col("Location").isin("ERROR", "UNKNOWN"),
                    "Unknown").otherwise(fs.col("Location")))
df.select("Location").distinct().display()

In [0]:
# Normalize error values handling ERROR/Unknown/null
df = df.withColumn("Transaction_Date", 
                    fs.when(fs.col("Transaction_Date").isNull() |
                    fs.col("Transaction_Date").isin("ERROR", "UNKNOWN"),
                    "Unknown").otherwise(fs.col("Transaction_Date")))
df.select("Transaction_Date").distinct().display()

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("`cafe-sales`.silver.cafe_sales_info")